# 02 - Baseline Modeling: Logistic Regression

This notebook establishes an interpretable classification baseline using the processed prediction dataset. It uses a stratified train/test split, fits preprocessing only on the training partition, and reports holdout metrics.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import mlflow
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go

from churn_ml.models.evaluate_model import classification_metrics

RANDOM_STATE = 42
TARGET_COLUMN = "Churn Value"
CHURN_THRESHOLD = None  # Set a value from 0 to 1 to override the training churn-rate threshold.
# CHURN_THRESHOLD = 0.5  # Set a value from 0 to 1 to override the training churn-rate threshold.

def find_project_file(relative_path):
    for directory in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        candidate = directory / relative_path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not find {relative_path} from {Path.cwd()} or its parent directories.")

DATA_PATH = find_project_file(Path("data/processed/prediction_df_logistic_regression.csv"))
VALUE_DATA_PATH = find_project_file(Path("data/raw/Telco_customer_churn.csv"))
PROJECT_ROOT = DATA_PATH.parents[2]
MLFLOW_EXPERIMENT_NAME = "telco-churn-modeling"
mlflow.set_tracking_uri("sqlite:///" + (PROJECT_ROOT / "mlflow.db").as_posix())
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

model_df = pd.read_csv(DATA_PATH)
value_df = pd.read_csv(VALUE_DATA_PATH, usecols=["CustomerID", TARGET_COLUMN, "CLTV"])
assert model_df.columns[-1] == TARGET_COLUMN, "The target must be the final column."

X = model_df.drop(columns=TARGET_COLUMN)
y = model_df[TARGET_COLUMN]
if len(value_df) != len(model_df) or not value_df[TARGET_COLUMN].equals(y):
    raise ValueError("Raw CLTV values are not aligned with the processed modeling data.")
customer_ltv = value_df.set_index("CustomerID")["CLTV"].rename("predicted_ltv_if_retained")
customer_ids = value_df["CustomerID"]
X_train, X_test, y_train, y_test, ltv_train, ltv_test, customer_id_train, customer_id_test = train_test_split(
    X, y, customer_ltv, customer_ids, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
decision_threshold = float(y_train.mean()) if CHURN_THRESHOLD is None else float(CHURN_THRESHOLD)
if not 0 < decision_threshold < 1:
    raise ValueError("CHURN_THRESHOLD must be between 0 and 1.")

print(f"Training rows: {len(X_train):,}; test rows: {len(X_test):,}")
print(f"Churn rate — train: {y_train.mean():.1%}; test: {y_test.mean():.1%}")
print(f"Prediction threshold: {decision_threshold:.1%}")

2026/07/04 07:30:54 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/07/04 07:30:54 INFO mlflow.store.db.utils: Updating database tables
2026/07/04 07:30:56 INFO mlflow.tracking.fluent: Experiment with name 'telco-churn-modeling' does not exist. Creating a new experiment.


Training rows: 5,634; test rows: 1,409
Churn rate — train: 26.5%; test: 26.5%
Prediction threshold: 26.5%


## Train the baseline

Median imputation handles the missing `Total Charges` values. Scaling places numeric features on comparable ranges, and balanced class weights account for the lower churn prevalence.

In [2]:
logistic_model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=1_000, class_weight="balanced", random_state=RANDOM_STATE)),
    ]
)
logistic_model.fit(X_train, y_train)

y_proba = logistic_model.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= decision_threshold).astype(int)
baseline_metrics = pd.Series(classification_metrics(y_test, y_pred, y_proba), name="logistic_regression")
display(baseline_metrics.to_frame())

confusion = confusion_matrix(y_test, y_pred, labels=[0, 1])
fig = go.Figure(go.Heatmap(
    z=confusion, x=["Predicted: no churn", "Predicted: churn"],
    y=["Actual: no churn", "Actual: churn"],
    colorscale="Blues", text=confusion, texttemplate="%{text}",
    colorbar={"title": "Customers"},
))
fig.update_layout(title=f"Logistic Regression Confusion Matrix (threshold = {decision_threshold:.1%})")
fig.show()

,logistic_regression
accuracy,0.647977
precision,0.425245
recall,0.927807
f1,0.583193
pr_auc,0.639283
roc_auc,0.846438


## Interpret coefficients

A positive coefficient increases the model's churn prediction, conditional on the other predictors. Coefficients are useful for interpretation but do not establish causality.

In [3]:
coefficient_summary = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": logistic_model.named_steps["classifier"].coef_.ravel(),
})
coefficient_summary["odds_ratio"] = np.exp(coefficient_summary["coefficient"])
coefficient_summary.reindex(
    coefficient_summary["coefficient"].abs().sort_values(ascending=False).index
).head(15)

,feature,coefficient,odds_ratio
2,Tenure Months,-0.755047,0.469989
1,Dependents,-0.693409,0.499869
20,Contract_Two year,-0.317969,0.727625
18,Contract_Month-to-month,0.316916,1.372887
8,Internet Service_Fiber optic,0.265465,1.304038
4,Paperless Billing,0.165429,1.179899
7,Internet Service_DSL,-0.161622,0.850762
10,Online Security_No,0.141956,1.152526
0,Partner,0.137898,1.147859
25,Streaming Service,0.135786,1.145437


# Insights

The baseline model uses the **training-set churn rate as its default classification threshold**. In this run, the training churn rate is **26.5%**, so customers with predicted churn probability at or above 26.5% are classified as likely to churn. Set `CHURN_THRESHOLD` to a value from 0 to 1 if a different operating point is needed.

On the holdout test set, the logistic model achieved:

| Threshold | Accuracy | Precision | Recall | F1 | ROC AUC | PR AUC |
|---|---:|---:|---:|---:|---:|---:|
| **26.5%** | 0.648 | 0.425 | 0.928 | 0.583 | 0.846 | 0.639 |

This threshold favors recall over precision. The model identifies about **92.8%** of actual churners, but only about **42.5%** of customers flagged for churn actually churn. That tradeoff may be appropriate when missed churners are costly and outreach is relatively inexpensive, but it will also create more false-positive retention outreach than a higher threshold such as 0.50.

A ROC AUC of **0.846** and PR AUC of **0.639** indicate useful ranking signal for a transparent baseline model. XGBoost and TabFM may improve ranking or retention-targeting value, but the largest future gains will likely come from richer inputs such as service-quality/outage history, support contacts and wait times, price changes, payment failures, usage or engagement trends, and recent plan changes.

# Expected Value of Retention Targeting

This evaluation retrieves the raw dataset's `CLTV` value for every holdout-test customer and treats it as that customer's **predicted lifetime value if retained**. `CLTV` is deliberately not a churn-model feature; it is used only after prediction to prioritize outreach.

The expected net value for each customer is calculated as:

$$P(\text{churn}) \times 10\% \times \text{predicted LTV if retained} - \$20 - (40\% \times \$500)$$

Assumptions: outreach costs **$20** for every targeted customer. The offer costs **$500** only when it is accepted; this scenario assumes a **40% offer-acceptance rate among targeted customers**, including customers who would have stayed without outreach. Its expected cost is therefore $200 per target. The 10% retention uplift is a separate scenario assumption: it represents the share of would-be churners saved by the intervention. Neither assumption is estimated by the churn model, so replace them when campaign data becomes available. The top 100 are selected by expected net value—not merely by churn probability—so high-value customers are prioritized.

In [4]:
OUTREACH_COST = 20
OFFER_COST = 500
RETENTION_UPLIFT = 0.10
OFFER_ACCEPTANCE_RATE = 0.40
TARGET_COUNT = 100

targeting_candidates = pd.DataFrame({
    "CustomerID": customer_id_test.to_numpy(),
    "predicted_churn_probability": y_proba,
    "predicted_ltv_if_retained": ltv_test.to_numpy(),
})
targeting_candidates["expected_value_before_cost"] = (
    targeting_candidates["predicted_churn_probability"]
    * RETENTION_UPLIFT
    * targeting_candidates["predicted_ltv_if_retained"]
)
targeting_candidates["expected_offer_cost"] = OFFER_COST * OFFER_ACCEPTANCE_RATE
targeting_candidates["campaign_cost"] = OUTREACH_COST + targeting_candidates["expected_offer_cost"]
targeting_candidates["expected_net_value"] = (
    targeting_candidates["expected_value_before_cost"]
    - targeting_candidates["campaign_cost"]
)

top_100_targets = (
    targeting_candidates
    .sort_values("expected_net_value", ascending=False)
    .head(TARGET_COUNT)
    .reset_index(drop=True)
)

targeting_summary = pd.DataFrame({
    "customers_targeted": [len(top_100_targets)],
    "expected_value_before_cost": [top_100_targets["expected_value_before_cost"].sum()],
    "outreach_cost": [OUTREACH_COST * len(top_100_targets)],
    "expected_offer_cost": [top_100_targets["expected_offer_cost"].sum()],
    "campaign_cost": [top_100_targets["campaign_cost"].sum()],
    "expected_net_value": [top_100_targets["expected_net_value"].sum()],
})
display(targeting_summary.style.format({
    "expected_value_before_cost": "${:,.2f}",
    "outreach_cost": "${:,.2f}",
    "expected_offer_cost": "${:,.2f}",
    "campaign_cost": "${:,.2f}",
    "expected_net_value": "${:,.2f}",
}))
top_100_targets.style.format({
    "predicted_churn_probability": "{:.1%}",
    "predicted_ltv_if_retained": "${:,.0f}",
    "expected_value_before_cost": "${:,.2f}",
    "expected_offer_cost": "${:,.2f}",
    "campaign_cost": "${:,.2f}",
    "expected_net_value": "${:,.2f}",
})

,customers_targeted,expected_value_before_cost,outreach_cost,expected_offer_cost,campaign_cost,expected_net_value
0,100,"$45,555.11","$2,000.00","$20,000.00","$22,000.00","$23,555.11"


,CustomerID,predicted_churn_probability,predicted_ltv_if_retained,expected_value_before_cost,expected_offer_cost,campaign_cost,expected_net_value
0,0295-PPHDO,92.7%,"$5,962",$552.51,$200.00,$220.00,$332.51
1,5178-LMXOP,94.3%,"$5,795",$546.70,$200.00,$220.00,$326.70
2,2865-TCHJW,92.3%,"$5,808",$536.04,$200.00,$220.00,$316.04
3,1320-HTRDR,90.1%,"$5,948",$535.70,$200.00,$220.00,$315.70
4,8361-LTMKD,89.2%,"$5,839",$520.91,$200.00,$220.00,$300.91
5,5228-EXCET,86.5%,"$5,964",$515.71,$200.00,$220.00,$295.71
6,7180-PISOG,92.7%,"$5,554",$514.87,$200.00,$220.00,$294.87
7,1628-BIZYP,89.3%,"$5,754",$513.90,$200.00,$220.00,$293.90
8,8541-QVFKM,88.6%,"$5,777",$511.86,$200.00,$220.00,$291.86
9,3716-BDVDB,87.6%,"$5,795",$507.46,$200.00,$220.00,$287.46


## MLflow tracking

This section records the lightweight experiment evidence for the baseline notebook: threshold policy, logistic-regression configuration, holdout metrics, and retention-targeting outputs. It logs scalar metrics and small tabular artifacts only.

In [5]:
with mlflow.start_run(run_name="02_logistic_regression_baseline"):
    mlflow.set_tags({
        "notebook": "02_baseline_modeling_logistic_regression.ipynb",
        "model_family": "logistic_regression",
        "stage": "baseline_modeling",
    })
    mlflow.log_params({
        "random_state": RANDOM_STATE,
        "target_column": TARGET_COLUMN,
        "data_path": str(DATA_PATH.relative_to(PROJECT_ROOT)),
        "threshold_policy": "training_churn_rate" if CHURN_THRESHOLD is None else "manual",
        "decision_threshold": decision_threshold,
        "train_churn_rate": float(y_train.mean()),
        "test_churn_rate": float(y_test.mean()),
        "model_max_iter": logistic_model.named_steps["classifier"].max_iter,
        "model_class_weight": logistic_model.named_steps["classifier"].class_weight,
        "imputer_strategy": logistic_model.named_steps["imputer"].strategy,
        "scaler": type(logistic_model.named_steps["scaler"]).__name__,
    })
    mlflow.log_metrics({f"holdout_{key}": float(value) for key, value in baseline_metrics.to_dict().items()})
    mlflow.log_metrics({
        "targeting_expected_net_value": float(targeting_summary.loc[0, "expected_net_value"]),
        "targeting_expected_value_before_cost": float(targeting_summary.loc[0, "expected_value_before_cost"]),
        "targeting_campaign_cost": float(targeting_summary.loc[0, "campaign_cost"]),
        "targeting_customers_targeted": int(targeting_summary.loc[0, "customers_targeted"]),
    })
    mlflow.log_table(baseline_metrics.reset_index().rename(columns={"index": "metric", "logistic_regression": "value"}), "tables/holdout_metrics.json")
    mlflow.log_table(coefficient_summary.sort_values("coefficient", key=lambda s: s.abs(), ascending=False), "tables/coefficient_summary.json")
    mlflow.log_table(targeting_summary, "tables/targeting_summary.json")
    mlflow.log_table(top_100_targets, "tables/top_100_targets.json")

print(f"Logged MLflow run to {PROJECT_ROOT / 'mlflow.db'}")

Logged MLflow run to W:\Workstation ExtDrive\007 Data Science\001 Data Science Training\2026_016 ML Churn Model End to End\mlflow.db
